In [ ]:
!pip -q uninstall -y cudf-cu12 pylibcudf-cu12


In [ ]:
!pip -q install -U "transformers>=4.44.2" "datasets>=2.20.0" "accelerate>=0.34.0" \
                 "peft>=0.12.0" "bitsandbytes>=0.43.1" "evaluate>=0.4.2" "einops" "sentencepiece"

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import pandas as pd
from transformers import IntervalStrategy, TrainingArguments as HFTrainingArguments

In [ ]:
import os, math, json, random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Any

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, default_data_collator, set_seed
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
PROJECT_DIR = Path("/content/drive/My Drive/CUIT")
TRAIN_JSONL = PROJECT_DIR / "train.jsonl"
VAL_JSONL   = PROJECT_DIR / "validation.jsonl"

In [ ]:
BASE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

In [ ]:
# assert TRAIN_JSONL.exists(),   f"Missing: {TRAIN_JSONL}"

In [ ]:
SEED = 42
set_seed(SEED)

In [ ]:
# Context & batching
MAX_LEN = 2048 # if T4 set 1024
BATCH_SIZE = 3
GRAD_ACCUM = 16 # effective batch = BATCH_SIZE * GRAD_ACCUM * num_gpus
EVAL_BATCH_SIZE = 1

In [ ]:
# QLoRA settings
USE_4BIT = True  # set False to train in fp16/bf16 without 4-bit quantization
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    # Common Qwen proj names
    "q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"
]

In [ ]:
# Training schedule
EPOCHS = 6 # ++
LR = 8e-5 # ++
WARMUP_RATIO = 0.1
LOG_STEPS = 10
SAVE_STEPS = 500
EVAL_STEPS = 500

In [ ]:
# Output
OUT_DIR = PROJECT_DIR / "qwen3_4b_instruct_qlora_out_v7"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
raw_ds = load_dataset(
    "json",
    data_files={"train": str(TRAIN_JSONL), "validation": str(VAL_JSONL)},
)

raw_ds

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

# Causal LM conventions
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.eos_token, tokenizer.pad_token, tokenizer.pad_token_id

In [ ]:
from typing import Dict, List, Any

def build_example(messages: List[Dict[str,str]]) -> Dict[str, List[int]]:
    # prefix
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True,
    )

    # assistant text (+EOS)
    asst_ids = tokenizer(messages[-1]["content"], add_special_tokens=False).input_ids
    if not asst_ids or asst_ids[-1] != tokenizer.eos_token_id:
        asst_ids = asst_ids + [tokenizer.eos_token_id]

    input_ids = prompt_ids + asst_ids
    labels    = [-100] * len(prompt_ids) + asst_ids[:]

    # truncate to MAX_LEN (left)
    if len(input_ids) > MAX_LEN:
        cut = len(input_ids) - MAX_LEN
        input_ids = input_ids[cut:]
        labels    = labels[cut:]

    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

def preprocess(example: Dict[str, Any]) -> Dict[str, List[int]]:
    msgs = example["messages"]
    # trim to last assistant if it is ending with user prompt
    if msgs[-1]["role"] != "assistant":
        for i in range(len(msgs)-1, -1, -1):
            if msgs[i]["role"] == "assistant":
                msgs = msgs[:i+1]
                break
    return build_example(msgs)

proc_train = raw_ds["train"].map(preprocess, remove_columns=raw_ds["train"].column_names, desc="Tokenizing train")
proc_val   = raw_ds["validation"].map(preprocess, remove_columns=raw_ds["validation"].column_names, desc="Tokenizing val")

len(proc_train), len(proc_val), proc_train[0].keys()

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding="longest",# pad per-batch to the longest example
    label_pad_token_id=-100, # pad as -100
    return_tensors="pt",
    pad_to_multiple_of=8,
)

In [ ]:
import transformers
from transformers import TrainingArguments as HFTrainingArguments
print("transformers =", transformers.__version__)
print("TrainingArguments module =", HFTrainingArguments.__module__)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

can_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if can_bf16 else torch.float16

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_cfg,
    torch_dtype=compute_dtype,
)

# disable cache for training
model.config.use_cache = False

# gradient checkpointing and prepare for k-bit
model.gradient_checkpointing_enable()
from peft import prepare_model_for_kbit_training
try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
except TypeError:
    model = prepare_model_for_kbit_training(model)
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES, # ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()   # sanity: LoRA params should require_grad=True


In [ ]:
steps_per_epoch = (len(proc_train) + (BATCH_SIZE*GRAD_ACCUM) - 1) // (BATCH_SIZE*GRAD_ACCUM)
print("steps_per_epoch =", steps_per_epoch)

In [ ]:
# obtaiin optimizer steps, not micro-batches
num_gpus = max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(proc_train) / (BATCH_SIZE * GRAD_ACCUM * num_gpus))
print("steps_per_epoch =", steps_per_epoch)

#  eval_steps is steps_per_epoch/4 floor eval 4 times
def nearest_divisor(n: int, target: int) -> int:
    divs = [d for d in range(1, n + 1) if n % d == 0]
    # if n < target, just use 1
    return min(divs, key=lambda d: abs(d - target))

raw_target = max(1, steps_per_epoch // 4)
eval_steps = nearest_divisor(steps_per_epoch, raw_target)
print(f"Eval every {eval_steps} steps (~4×/epoch)")

In [ ]:
training_args = HFTrainingArguments(
    output_dir=str(OUT_DIR),

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    learning_rate=LR,
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,

    # show train loss
    logging_strategy=IntervalStrategy.STEPS,
    logging_steps=max(1, eval_steps // 5),   #log 4 times between evals
    logging_first_step=True,

    # 4 eval evaluate per epoch
    eval_strategy=IntervalStrategy.STEPS,
    eval_steps=eval_steps,

    # save each epoch
    save_strategy=IntervalStrategy.STEPS,
    save_steps=steps_per_epoch,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    # stability
    weight_decay=0.01,
    max_grad_norm=1.0,

    # performance / memory
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    optim="paged_adamw_32bit" if USE_4BIT else "adamw_torch",
    ddp_find_unused_parameters=False,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
from transformers import TrainerCallback, EarlyStoppingCallback

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=proc_train,
    eval_dataset=proc_val,
    processing_class=tokenizer,   # tokenizer=tokenizer
    data_collator=data_collator,
)
trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=2))

In [ ]:
# start or resume training
import glob
resume = True  # change here for new training
ckpts = sorted(glob.glob(str(OUT_DIR / "checkpoint-*")))
resume_ckpt = ckpts[-1] if (resume and ckpts) else None

train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
trainer.save_state()  # save states to OUT_DIR

# eval
import math
metrics = trainer.evaluate()
val_loss = metrics.get("eval_loss")
ppl = (math.exp(val_loss) if val_loss is not None else None)
print({"eval_loss": val_loss, "perplexity": ppl})

# saving
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)


In [ ]:
OUT_DIR = Path(trainer.args.output_dir)
state_path = OUT_DIR / "trainer_state.json"

with state_path.open("r", encoding="utf-8") as f:
    st = json.load(f)

rows = []
for e in st.get("log_history", []):
    step  = e.get("step", e.get("global_step"))
    loss  = e.get("loss", e.get("train_loss"))
    eval_ = e.get("eval_loss")
    rows.append({
        "step": step,
        "epoch": e.get("epoch"),
        "train_loss": loss,
        "eval_loss": eval_,
        "perplexity": (math.exp(eval_) if eval_ is not None else None),
    })

df = pd.DataFrame(rows).sort_values("step").drop_duplicates(subset=["step"], keep="last")
print(df.tail(20)) # show last 20 entries
print(df[df.eval_loss.notna()].tail(10))  # last evals only

In [ ]:
model.eval()

# inference
def chat_generate(messages: List[Dict[str,str]], max_new_tokens=256, temperature=0.7, top_p=0.9):
    # byukd chat prompt from messages
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    out = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return out.strip()

# generate from val set
sample = raw_ds["validation"][0]["messages"]
print("USER:", [m["content"] for m in sample if m["role"]=="user"][-1])
print("ASSISTANT (gen):", chat_generate(sample[:-1]))

In [ ]:
# save the adapter
ADAPTER_DIR = PROJECT_DIR / "qwen3_lora_adapter_v7"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("LoRA adapter saved to:", ADAPTER_DIR)